# Практическая работа №1: Классификация изображений CNN
Бахорина Ольга Александровна  
Группа R4176c  
1 курс магистратуры  
Дисциплина «Прикладной искусственный интеллект»

Полное обучение и итоговая оценка моделей выполнены в облачной среде Kaggle с использованием GPU.

## Постановка и объём
CNN/ResNet18; четыре конфигурации. Ответы после экспериментов: архитектура ResNet18; работа и параметры; CNN против MLP.
Основание: docs/requirements.md и docs/specification.md. Исходные учебные ноутбуки сохранены отдельно.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'aai').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Откройте ноутбук внутри клонированного проекта')
sys.path.insert(0, str(ROOT))
from aai.common import services
from aai.common.config import load_config
from aai.common.data import load_dataset
from aai.common.artifacts import read_json
print(sys.version)

In [ ]:
PRACTICE = 1
mode = 'documented_results'  # итоговые метрики получены на Kaggle и не пересчитываются локально
config = load_config(ROOT / 'practice_01/configs/cnn_a.yaml')
KAGGLE_FULL_NOTEBOOK = 'приватный Kaggle Notebook: полное обучение'
KAGGLE_FINAL_NOTEBOOK = 'приватный Kaggle Notebook: итоговая оценка'
print(KAGGLE_FULL_NOTEBOOK)
print(KAGGLE_FINAL_NOTEBOOK)

## Источник и аудит
Следующая ячейка читает подготовленные manifests. Данные и веса не скачиваются автоматически. При отсутствии данных остановитесь и выполните отдельный этап подготовки.

In [ ]:
dataset = load_dataset(config)
print(dataset['source'])
print(dataset['counts'])
print(dataset['preprocessing'])
print(dataset['rows'][:2])

In [ ]:
from aai.p1.models import build_model
architecture = build_model(config, dataset['mapping'], initialize=False)
models = architecture if isinstance(architecture, dict) else {'model': architecture.model if hasattr(architecture, 'predictor') else architecture}
print({name: {'parameters': sum(p.numel() for p in model.parameters()), 'trainable': sum(p.numel() for p in model.parameters() if p.requires_grad)} for name, model in models.items()})
# Только схема архитектуры; итоговые веса и метрики берутся из selection.

## Матрица опытов и выбор моделей
На Kaggle T4×2 завершены две конфигурации CNN с нуля и две конфигурации ResNet18. По validation выбраны p1_cnn_a (accuracy 0.936333, macro-F1 0.936319) и p1_resnet_a (accuracy 0.999667, macro-F1 0.999667). Test не использовался при выборе.

In [ ]:
if mode != 'documented_results':
    raise ValueError('Этот итоговый ноутбук документирует подтверждённые результаты; запуск обучения выполняется отдельным Kaggle Notebook')

In [ ]:
selected = {
    'cnn': {'run_id': '20260914T033444204250Z_p1_cnn_a_fce25d5e', 'val_accuracy': 0.9363333333333334, 'val_macro_f1': 0.9363187336642439},
    'resnet18': {'run_id': '20260914T035210722298Z_p1_resnet_a_d0a47ce7', 'val_accuracy': 0.9996666666666667, 'val_macro_f1': 0.9996666645833202},
}
selected

In [ ]:
final_evaluation = {
    'source': 'Итоговая оценка в приватном Kaggle Notebook; не пересчитывается локально',
    'cnn': {'test_accuracy': 0.9376666666666666, 'test_macro_f1': 0.9374514356959985, 'test_loss': 0.2263465506931146},
    'resnet18': {'test_accuracy': 0.999, 'test_macro_f1': 0.998999154129973, 'test_loss': 0.004157620379325349},
    'checkpoint_verification': 'verified; 4 passed',
}
final_evaluation

In [ ]:
from pathlib import Path
history_paths = {
    'cnn': ROOT / 'practice_01/artifacts/kaggle/synchronized/p01-full-v1-b0daa7d02677-f82f6d8a/runs/20260914T033444204250Z_p1_cnn_a_fce25d5e/history.csv',
    'resnet18': ROOT / 'practice_01/artifacts/kaggle/synchronized/p01-full-v1-b0daa7d02677-f82f6d8a/runs/20260914T035210722298Z_p1_resnet_a_d0a47ce7/history.csv',
}
assert all(path.is_file() for path in history_paths.values())
history_paths

## Анализ ошибок и выводы
CNN: test accuracy 0.937667, macro-F1 0.937451; ошибки наиболее заметны для визуально близких классов. ResNet18: test accuracy 0.999000, macro-F1 0.998999. ResNet18 выбрана по validation и показала лучший результат на test; вывод относится только к данному датасету и протоколу.

ResNet18 использует остаточные блоки и shortcut-соединения, что облегчает обучение глубокой сети. CNN сохраняет пространственную структуру изображения и совместно использует веса, тогда как MLP после развёртывания пикселей этой предпосылки лишён и требует больше параметров.

## Источники и воспроизведение
Источники задания: ../docs/requirements.md; технические решения: ../docs/specification.md. Источник данных: https://www.kaggle.com/datasets/misrakahmed/vegetable-image-dataset.
Полное обучение и итоговая оценка сохранены в приватных Kaggle Notebook. Конфигурации, идентификаторы запусков и проверка checkpoint зафиксированы в артефактах проекта.
Команды повторной оценки и сборки PDF: README.md этой практики. Тексты выводов — ../reports/src/. Официальный перечень тем зачёта указан в приложении Б ТЗ.